In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader


In [2]:
# torch.cuda.empty_cache()

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

NUM_CLASSES = 11
BATCH_SIZE = 32
LOCAL_EPOCHS = 15
# ROUNDS = 10
LR = 1e-4

TRAIN_DIR = "TRAIN"
VAL_DIR = "VAL"
TEST_DIR = "TEST"

cuda


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((384,384)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((384,384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [5]:
# Loading ResNet-50 Model
# def get_model():
#     model = models.resnet50(pretrained=True)
#     model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading Swin_v2_Tiny or Swin_v2_Base Model
# def get_model():
#     model = models.swin_v2_b(weights=models.Swin_V2_B_Weights.IMAGENET1K_V1)
#     model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading ViT_b_16
# def get_model():
#     model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
#     model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
#     return model.to(DEVICE)

In [6]:
# Loading ConvNext Base
# def get_model():
#     model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
#     model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
#     return model.to(DEVICE)

# Loading EfficientNet_V2_S
def get_model():
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    return model.to(DEVICE)

In [7]:
def get_client_loader(client_name):
    client_path = os.path.join(TRAIN_DIR, client_name)
    print(f"Loading data from {client_path}")
    dataset = datasets.ImageFolder(client_path, transform=train_transform)
    print(f"Number of training data instances: {len(dataset)}")
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)


In [8]:
val_loader = DataLoader(
    datasets.ImageFolder(VAL_DIR, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    datasets.ImageFolder(TEST_DIR, transform=eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [9]:
from tqdm import tqdm

def local_train(model, loader):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    # criterion = FocalLoss(alpha=class_weights, gamma=2.0)

    for epoch in range(LOCAL_EPOCHS):
        epoch_loss = 0.0

        progress_bar = tqdm(loader, desc=f"Local Epoch [{epoch+1}/{LOCAL_EPOCHS}]", leave=False)

        for x, y in progress_bar:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)

            # if isinstance(outputs, tuple) or hasattr(outputs, "logits"):
            #     logits = outputs.logits
            #     aux_logits = outputs.aux_logits
            #     loss = criterion(logits, y) + 0.4 * criterion(aux_logits, y)
            # else:
            #     loss = criterion(outputs, y)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        print(f"Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    return model.state_dict()

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro')  
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    return acc, precision, recall, f1

In [11]:
def average_models(models):

    avg_model = copy.deepcopy(models[0])
    avg_dict = avg_model.state_dict()

    for key in avg_dict.keys():

        tensors = [m.state_dict()[key] for m in models]

        # If tensor is float → average
        if tensors[0].dtype in [torch.float32, torch.float64, torch.float16]:
            avg_dict[key] = torch.stack(tensors, dim=0).mean(dim=0)

        # If tensor is int (e.g., num_batches_tracked) → copy from first model
        else:
            avg_dict[key] = tensors[0]

    avg_model.load_state_dict(avg_dict)
    return avg_model

def greedy_soup(models, val_loader, val_scores):

    # val_scores = [evaluate(m, val_loader) for m in models]
    sorted_indices = sorted(range(len(models)), key=lambda i: val_scores[i], reverse=True)

    soup_model = copy.deepcopy(models[sorted_indices[0]])
    best_score = val_scores[sorted_indices[0]]
    selected = [sorted_indices[0]]

    for idx in sorted_indices[1:]:
        temp_model = average_models([soup_model, models[idx]])
        score = evaluate(temp_model, val_loader)[0]

        # print(type(score))
        # print(score)
        # print(type(best_score))

        if score > best_score:
            soup_model = temp_model
            best_score = score
            selected.append(idx)

    return soup_model, selected, val_scores



In [12]:
def reliability_weighted_avg(models, reliabilities, selected_indices):

    weights = torch.tensor([reliabilities[i] for i in selected_indices])
    weights = weights / weights.sum()

    global_model = copy.deepcopy(models[selected_indices[0]])
    global_dict = global_model.state_dict()

    for key in global_dict.keys():
        weighted_params = torch.stack([
            weights[i] * models[selected_indices[i]].state_dict()[key]
            for i in range(len(selected_indices))
        ], dim=0)

        global_dict[key] = weighted_params.sum(dim=0)

    global_model.load_state_dict(global_dict)
    return global_model

In [13]:
clients = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, d)) and not d.startswith(".")
])

print(f"Found clients: {clients}")

Found clients: ['client_01', 'client_02', 'client_03', 'client_04', 'client_05']


In [14]:
CKPT_DIR = "EfficientNet_V2_S_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

client_models = []
client_val_accs = []
client_val_f1s = []

# Train 5 clients
for client in clients:
    print(client)
    
    client_loader = get_client_loader(client)

    model = get_model()
    model_dict = local_train(model, client_loader)
    client_val_acc, client_val_pre, client_val_recall, client_val_f1 = evaluate(model, val_loader)
    print(f"{client} Validation Accuracy: {client_val_acc:.4f}")
    print(f"{client} Validation Precision: {client_val_pre:.4f}")
    print(f"{client} Validation Recall: {client_val_recall:.4f}")
    print(f"{client} Validation F1 Score: {client_val_f1:.4f}")
    print("\n ============================================== \n")
        
    torch.save({
        "client_name": client,
        "model_state_dict": model_dict,
        "val_accuracy": client_val_acc,
        "val_precision": client_val_pre,
        "val_recall": client_val_recall,
        "val_f1": client_val_f1
    }, os.path.join(CKPT_DIR, f"{client}.pth"))

    client_models.append(model)
    client_val_accs.append(client_val_acc)
    client_val_f1s.append(client_val_f1)

client_01
Loading data from TRAIN/client_01
Number of training data instances: 99285


Epoch 1 Loss: 0.4378


Epoch 2 Loss: 0.1548


Epoch 3 Loss: 0.0991


Epoch 4 Loss: 0.0793


...

Epoch 11 Loss: 0.0487


Epoch 12 Loss: 0.0474


Epoch 13 Loss: 0.0473


Epoch 14 Loss: 0.0432


Epoch 15 Loss: 0.0416
client_05 Validation Accuracy: 0.7985
client_05 Validation Precision: 0.7944
client_05 Validation Recall: 0.7951
client_05 Validation F1 Score: 0.7933




In [15]:
len(client_models)

5

In [16]:
print([f"{acc:.4f}" for acc in client_val_accs])
print("---------------------------")
print([f"{f1:.4f}" for f1 in client_val_f1s])

['0.8075', '0.7971', '0.8000', '0.7950', '0.7985']
---------------------------
['0.8019', '0.7941', '0.7985', '0.7926', '0.7933']


In [17]:
# Greedy Soup
soup_model, selected_indices, reliabilities = greedy_soup(client_models, val_loader, client_val_accs)

# Reliability Weighted Global Model
global_model = reliability_weighted_avg(
    client_models,
    reliabilities,
    selected_indices
)

acc, precision, recall, f1 = evaluate(global_model, test_loader)


print(f"Global Model Test Accuracy: {acc:.4f}")
print(f"Global Model Test Precision: {precision:.4f}")
print(f"Global Model Test Recall: {recall:.4f}")
print(f"Global Model Test F1 Score: {f1:.4f}")

print("Selected Clients:", selected_indices)

Global Model Test Accuracy: 0.8161
Global Model Test Precision: 0.8149
Global Model Test Recall: 0.8157
Global Model Test F1 Score: 0.8122
Selected Clients: [0, 4]


In [ ]:
# FedAvg baseline
def fedavg(models):

    global_model = copy.deepcopy(models[0])
    global_dict = global_model.state_dict()

    for key in global_dict.keys():s

        tensors = [models[i].state_dict()[key] for i in range(len(models))]

        # Only average floating point tensors
        if tensors[0].dtype.is_floating_point:
            global_dict[key] = torch.stack(tensors, dim=0).mean(dim=0)
        else:
            # For integer tensors (e.g., num_batches_tracked)
            global_dict[key] = tensors[0]

    global_model.load_state_dict(global_dict)
    return global_model

fedavg_model = fedavg(client_models)
fedavg_test_acc, fedavg_test_pre, fedavg_test_recall, fedavg_test_f1 = evaluate(fedavg_model, test_loader)
print("FedAvg Test Accuracy:", fedavg_test_acc)
print("FedAvg Test Precision:", fedavg_test_pre)
print("FedAvg Test Recall:", fedavg_test_recall)
print("FedAvg Test F1 Score:", fedavg_test_f1)

FedAvg Test Accuracy: 0.7892659163447001
FedAvg Test Precision: 0.8030746797849623
FedAvg Test Recall: 0.7879533392981938
FedAvg Test F1 Score: 0.7845866810368811
